# 🛡️ CyberRott v1.4 — Pipeline Unificado de Treinamento e Alinhamento por Reforço Online (GRPO + DoRA + PRM + TTC)

Arquitetura unificada de treinamento para o **CyberRott v1.1 / v1.2 / v1.3 / v1.4 (Phi-4-mini 3.82B / Multi-Backbone)** implementando o motor de **Group Relative Policy Optimization (GRPO)** com **Weight-Decomposed Low-Rank Adaptation (DoRA $r=64, \alpha=128$)**, **Step-Level Process Reward Model (PRM)**, **Test-Time Compute (TTC)** e **Janela de Contexto de 2048 Tokens** — a formulação matemática de RL do **DeepSeek-R1** — nativamente com Hugging Face `TRL` e `PEFT`.

```text
===================================================================================================
                       FLUXO DE TREINAMENTO E AUTO-EVOLUÇÃO (CYBERROTT v1.4)
===================================================================================================

+---------------------------------------------------------------------------------------------------+
| FASE 1: Dataset Consolidado Curado v1.4 (4.680 Amostras CoT Open-Source em ChatML)                |
| - Amostras ricas em verificação técnica (CWE, MITRE ATT&CK, Múltipla Escolha e Código)            |
| - 16x Golden Anchors Ponderadas + Scrubber de Recusas & CoT de 7 Estágios (<think>)               |
| - Carregamento direto via GitHub (data/cyberrott_final_train.jsonl) sem retrabalho                |
+---------------------------------------------------------------------------------------------------+
                                                  │
                                                  ▼
+---------------------------------------------------------------------------------------------------+
| FASE 2: Treinamento SFT Base DoRA v1.4 (Detecção Inteligente de Checkpoint no Drive)              |
| - Se o checkpoint SFT DoRA já existir no Drive/local, pula o retreino e carrega instantaneamente! |
| - Treinamento Supervisionado DoRA r=64, alpha=128, all-linear, Contexto=2048, BF16/TF32 na GPU    |
+---------------------------------------------------------------------------------------------------+
                                                  │
                                                  ▼
+---------------------------------------------------------------------------------------------------+
| FASE 3: Treinamento Online GRPO v1.4 (Group Relative Policy Optimization - DeepSeek-R1 Style)      |
| 1. In-Memory SFT Merge: Funde o DoRA SFT na memória em segundos -> π_ref com ZERO VRAM extra!    |
| 2. Geração em Grupo: G=8 rollouts exploratórios por prompt com vLLM colocate                      |
| 3. Suíte de 4 Recompensas Determinísticas Verificáveis (STaRVerifier + Step-Level PRM):          |
|    • cot_integrity_reward (0.30): <think> tags, 7 estágios analíticos, anti-recusa estrita        |
|    • technical_accuracy_reward (0.50): Gabarito exato de múltipla escolha e CWE/MITRE matches      |
|    • sandbox_code_reward (0.20): Compilação AST Python, gramática YARA e regras Sigma/YAML        |
|    • step_level_prm_reward (0.25): Verificação granular de consistência lógica em etapas          |
| 4. Otimização Sem Crítico (Critic-Free): Economiza ~15GB de VRAM, viabilizando A100 e L4          |
+---------------------------------------------------------------------------------------------------+
                                                  │
                                                  ▼
+---------------------------------------------------------------------------------------------------+
| FASE 4: Auditoria de Vulnerabilidades & Exportação Final (Tripartite Output)                      |
| - Avaliação com Raciocínio CoT (<think>) + Diagnóstico Técnico (FATO) + Correção em Sandbox       |
| - Sincronização automática do checkpoint GRPO v1.4 final para o Google Drive                      |
+---------------------------------------------------------------------------------------------------+
                                                  │
                                                  ▼
+---------------------------------------------------------------------------------------------------+
| FASE 5: Avaliação Formal de Benchmark WMDP-Cyber com Test-Time Compute (TTC v1.4)                 |
| - Auto-fusão BF16 pura e inferência em lote com motor vLLM em 2048 tokens                         |
| - Estratégia de Test-Time Compute: Best-of-8 (ou Majority Voting / PRM Weighted SC)               |
| - Geração e download automático de relatórios JSON de telemetria e acurácia formal                 |
+---------------------------------------------------------------------------------------------------+
                                                  │
                                                  ▼
+---------------------------------------------------------------------------------------------------+
| FASE 6: Publicação e Upload para o Hugging Face Hub (Model Card Oficial + Pesos Standalone)       |
| - Upload direto de safetensors standalone, Modelfile para Ollama e README.md formatado            |
+---------------------------------------------------------------------------------------------------+
```

In [ ]:
# 1. Instalação das dependências modernas (compatibilidade estrita com PyTorch 2.x e CUDA 12)
!pip install -q -U "trl>=0.16.0" "fastapi" "uvicorn" "pydantic"
!pip install -q "transformers>=4.46.0,<5.0.0" "peft>=0.13.0,<0.16.0" "accelerate>=1.0.0" "datasets>=3.0.0" "bitsandbytes>=0.44.0" "pyyaml" "vllm"

# 2. Diagnóstico de Aceleração de Hardware GPU (L4 / A100 / H100)
import os
import torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("=" * 70)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    major, minor = torch.cuda.get_device_capability(0)
    bf16_supported = torch.cuda.is_bf16_supported()

    # Ativação de TF32 para arquiteturas modernas (Ampere, Ada Lovelace, Hopper)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    tf32_status = 'Ativado com sucesso' if major >= 8 else 'Não disponível'
    print(f"Dispositivo GPU: {device_name}")
    print(f"VRAM Total:      {total_vram_gb:.1f} GB")
    print(f"Arquitetura:     Compute Capability {major}.{minor} (Ampere/Ada/Hopper: {major >= 8})")
    print(f"Suporte BF16:    {bf16_supported}")
    print(f"TF32 TensorCore: {tf32_status}")
else:
    print("GPU não detectada! Ative GPU em: Runtime -> Change runtime type -> Hardware accelerator.")
print("=" * 70)
!nvidia-smi


In [ ]:
import os
import sys

# 1. Montagem Opcional do Google Drive (Persistência Segura de Checkpoints)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DRIVE_DIR = "/content/drive/MyDrive/CyberRott_Checkpoints"
    os.makedirs(CHECKPOINT_DRIVE_DIR, exist_ok=True)
    print(f"[✓] Google Drive montado com sucesso em: {CHECKPOINT_DRIVE_DIR}")
except Exception:
    print("[*] Google Drive não montado (ambiente local ou execução independente).")

# 2. Localização e Sincronização do Repositório CyberRott
REPO_DIR = "/content/CyberRott"
if os.path.exists("/content"):
    if not os.path.exists(REPO_DIR):
        print("[*] Clonando repositório CyberRott no ambiente Colab...")
        !git clone https://github.com/johnknevescarrijo/CyberRott.git {REPO_DIR}
    else:
        print("[*] Repositório CyberRott já existente. Atualizando via git pull...")
        !git -C {REPO_DIR} pull origin main
    os.chdir(REPO_DIR)
elif os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull origin main
    os.chdir(REPO_DIR)

print(f"[✓] Diretório de Trabalho: {os.getcwd()}")
sys.path.insert(0, os.getcwd())

# 3. Configuração Segura do HF_TOKEN (Google Colab Secrets ou Variável de Ambiente)
try:
    from google.colab import userdata
    hf_secret = userdata.get("HF_TOKEN")
    if hf_secret:
        os.environ["HF_TOKEN"] = hf_secret
        print("[✓] HF_TOKEN carregado com sucesso do Google Colab Secrets.")
except Exception:
    pass

if "HF_TOKEN" in os.environ and os.environ["HF_TOKEN"]:
    print("[✓] HF_TOKEN configurado no ambiente.")
else:
    print("[*] HF_TOKEN não detectado nos Secrets (opcional para modelos públicos).")


---
## 📦 FASE 1: Dataset Consolidado v1.4 (`data/cyberrott_final_train.jsonl`)

O repositório já inclui por padrão o corpus consolidado `data/cyberrott_final_train.jsonl` (4.680 amostras / ~41 MB) com raciocínio analítico profundo de 7 estágios, metadados de verificação (CWE, MITRE ATT&CK, gabaritos de múltipla escolha e blocos de código) e as Âncoras de Ouro em `data/dataset_cyber_ready.jsonl`.

Se o arquivo já estiver presente no diretório clonado do GitHub, **ele é utilizado diretamente sem nenhum retrabalho**. Caso contrário, o pipeline de preparação v1.4 é disparado automaticamente.


In [ ]:
import os
import json
import urllib.request

DATASET_FILE = "data/cyberrott_final_train.jsonl"
ANCHOR_FILE = "data/dataset_cyber_ready.jsonl"

def count_samples(filepath):
    if not os.path.exists(filepath):
        return 0
    with open(filepath, "r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

# Checagem: se o dataset v1.4 completo já existir com >= 4500 amostras, usa diretamente
existing_samples = count_samples(DATASET_FILE)
if existing_samples >= 4500:
    print(f"[✓] Dataset v1.4 completo ({existing_samples:,} amostras / {os.path.getsize(DATASET_FILE)/(1024*1024):.2f} MB) já presente localmente via Git! Pulando geração...")
else:
    print(f"[*] Dataset local ausente ou incompleto ({existing_samples} amostras). Baixando versão consolidada do GitHub...")
    os.makedirs("data", exist_ok=True)
    raw_url = "https://raw.githubusercontent.com/johnknevescarrijo/CyberRott/main/data/cyberrott_final_train.jsonl"
    try:
        urllib.request.urlretrieve(raw_url, DATASET_FILE)
        print(f"[✓] Download do dataset final concluído: {count_samples(DATASET_FILE):,} amostras.")
    except Exception as e:
        print(f"[!] Erro no download direto ({e}). Executando pipeline data_prep.py com preset v1.4...")
        !python scripts/data_prep.py \
            --preset v1.4 \
            --anchor-file data/dataset_cyber_ready.jsonl \
            --output-file data/cyberrott_final_train.jsonl \
            --seed 42

# Validação e Estatísticas do Corpus
if os.path.exists(DATASET_FILE):
    size_mb = os.path.getsize(DATASET_FILE) / (1024 * 1024)
    total_samples = count_samples(DATASET_FILE)
    print("=" * 65)
    print("📊 DATASET DE TREINAMENTO V1.4 CARREGADO COM SUCESSO:")
    print(f"   • Caminho:          {DATASET_FILE}")
    print(f"   • Total de Amostras:{total_samples:,}")
    print(f"   • Tamanho em Disco: {size_mb:.2f} MB")
    print(f"   • Âncoras de Ouro:  {count_samples(ANCHOR_FILE)} amostras")
    print("=" * 65)


---
## 🚀 FASE 2: Checkpoint SFT Base DoRA v1.4 (`CyberRott-v1.4-SFT-DoRA`)

O modelo GRPO (Fase 3) necessita de um checkpoint supervisionado (`SFT_LORA`) para inicializar a política e servir de referência ($\pi_{\text{ref}}$). Na versão 1.4, introduzimos o **DoRA (Weight-Decomposed Low-Rank Adaptation)** com $r=64$ e $\alpha=128$, dobrando a capacidade expressiva do modelo e estabilizando a convergência.

### Verificação Inteligente de Checkpoint:
- Se você já treinou ou sincronizou o checkpoint do `SFT_LORA` v1.4 no **Google Drive** (em `/content/drive/MyDrive/CyberRott_Checkpoints/CyberRott-v1.4-SFT-DoRA`) ou no disco local (`./CyberRott-v1.4-SFT-DoRA`), ele será **detectado e reutilizado automaticamente, pulando o retreino SFT**!
- Se nenhum checkpoint v1.4 for encontrado, o script `scripts/train_colab.py` é executado com `--preset_v1_4` para treinar o SFT DoRA em `bfloat16`, salvando automaticamente uma cópia de segurança no Google Drive.


In [ ]:
import os
import shutil

# Verificação Inteligente de Checkpoint SFT DoRA v1.4
LOCAL_SFT_DIR = "./CyberRott-v1.4-SFT-DoRA"
CANDIDATE_PATHS = [
    LOCAL_SFT_DIR,
    "/content/drive/MyDrive/CyberRott_Checkpoints/CyberRott-v1.4-SFT-DoRA",
    "/content/drive/MyDrive/CyberRott_Checkpoints/CyberRott_v1_4_SFT_DoRA",
]

sft_checkpoint_found = None
for path in CANDIDATE_PATHS:
    if os.path.exists(path) and os.path.exists(os.path.join(path, "adapter_config.json")):
        sft_checkpoint_found = path
        break

if sft_checkpoint_found:
    print(f"[✓] Checkpoint SFT DoRA v1.4 prévio encontrado em: '{sft_checkpoint_found}'!")
    if sft_checkpoint_found != LOCAL_SFT_DIR:
        print(f"[*] Sincronizando adaptador para o diretório local '{LOCAL_SFT_DIR}'...")
        shutil.copytree(sft_checkpoint_found, LOCAL_SFT_DIR, dirs_exist_ok=True)
    print("[✓] Checkpoint SFT DoRA v1.4 pronto para fusão na Fase 3. RETREINO SFT PULADO COM SUCESSO!")
else:
    print("[*] Nenhum checkpoint SFT DoRA v1.4 prévio encontrado. Iniciando treinamento SFT v1.4 da Fase 2...")
    !export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True && python scripts/train_colab.py \
        --preset_v1_4 \
        --data_path data/cyberrott_final_train.jsonl \
        --output_dir ./CyberRott-v1.4-SFT-DoRA \
        --epochs 3 \
        --per_device_train_batch_size 2 \
        --gradient_accumulation_steps 8 \
        --max_seq_length 2048 \
        --torch_dtype bfloat16 \
        --attn_implementation sdpa \
        --learning_rate 2e-4

    # Backup automático no Google Drive
    if os.path.exists("/content/drive/MyDrive/CyberRott_Checkpoints") and os.path.exists(LOCAL_SFT_DIR):
        drive_dest = "/content/drive/MyDrive/CyberRott_Checkpoints/CyberRott-v1.4-SFT-DoRA"
        print(f"[*] Salvando backup do SFT DoRA v1.4 no Google Drive: {drive_dest}...")
        shutil.copytree(LOCAL_SFT_DIR, drive_dest, dirs_exist_ok=True)
        print("[✓] Backup do SFT DoRA v1.4 concluído com sucesso!")


---
## 🧠 FASE 3: Treinamento por Reforço Online via GRPO v1.4 (DeepSeek-R1 Style + DoRA + PRM + vLLM)

Substituindo o antigo ciclo offline (`scripts/train_star.py`), o **CyberRott v1.4** implementa **Group Relative Policy Optimization (GRPO)** — a mesma formulação matemática do DeepSeek-R1 — utilizando Hugging Face `TRL` (`GRPOTrainer` e `GRPOConfig`) com aceleração massiva via **vLLM**:

### Inovações Arquiteturais do GRPO v1.4:
1. **Fusão em Memória Instantânea ($\\pi_{\\text{ref}}$ com Zero VRAM Extra):**
   O checkpoint `CyberRott-v1.4-SFT-DoRA` da Fase 2 é fundido em tempo de execução na memória do processo em ~3 segundos, servindo como política de referência congelada sem carregar um segundo modelo completo.
2. **DoRA ($r=64, \\alpha=128$) na Política Ativa:**
   Decomposição de magnitude e direção para aprendizado por reforço de alta capacidade, permitindo ajustes direcionais finos nas matrizes de atenção e projeção MLP.
3. **Suíte de 4 Recompensas Determinísticas Verificáveis:**
   - `cot_integrity_reward` (0.30): Validação de 7 estágios analíticos, integridade da tag `<think>` e anti-recusa estrita;
   - `technical_accuracy_reward` (0.50): Gabarito exato de múltipla escolha e identificação correta de CWE/MITRE;
   - `sandbox_code_reward` (0.20): Execução e sintaxe segura em Python AST, regras YARA e Sigma/YAML;
   - `step_level_prm_reward` (0.25): Process Reward Model avaliando a consistência lógica passo a passo no raciocínio CoT.
4. **Motor vLLM Ativo com PagedAttention ($G=8$ rollouts paralelos):**
   Rollouts ultrarrápidos com vLLM server dedicado (FastAPI/NCCL weight synchronization), reduzindo o tempo de geração de ~16s/it para ~1.2s/it (~50-80 minutos totais na A100).
5. **Auto-Resume Resiliente:**
   Detecta automaticamente o último checkpoint salvo (`checkpoint-*`) no diretório ou Google Drive, permitindo continuar exatamente de onde parou sem perda de progresso.


In [ ]:
# Configuração de alocação de memória CUDA para prevenir fragmentação
import os
import sys
import time
import shutil
import urllib.request

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 1. Verificação Inteligente de Checkpoint Final GRPO v1.4
LOCAL_GRPO_DIR = "./CyberRott-v1.4-GRPO"
CANDIDATE_GRPO_PATHS = [
    LOCAL_GRPO_DIR,
    "/content/drive/MyDrive/CyberRott_Checkpoints/CyberRott-v1.4-GRPO",
    "/content/drive/MyDrive/CyberRott_Checkpoints/CyberRott_v1_4_GRPO",
]

grpo_checkpoint_found = None
for path in CANDIDATE_GRPO_PATHS:
    if os.path.exists(path) and os.path.exists(os.path.join(path, "adapter_config.json")):
        grpo_checkpoint_found = path
        break

if grpo_checkpoint_found:
    print(f"[✓] Checkpoint GRPO v1.4 prévio encontrado em: '{grpo_checkpoint_found}'!")
    if grpo_checkpoint_found != LOCAL_GRPO_DIR:
        print(f"[*] Sincronizando adaptador GRPO para o diretório local '{LOCAL_GRPO_DIR}'...")
        shutil.copytree(grpo_checkpoint_found, LOCAL_GRPO_DIR, dirs_exist_ok=True)
    print("[✓] Checkpoint GRPO v1.4 pronto para inferência e benchmark. RETREINO GRPO PULADO COM SUCESSO!")
else:
    print("[*] Preparando ambiente de treinamento por reforço online via GRPO v1.4...")

    # 2. Inicialização e Diagnóstico do Servidor vLLM (Geração Ultra-Acelerada de Rollouts)
    VLLM_HOST = "127.0.0.1"
    VLLM_PORT = 8000
    HEALTH_URL = f"http://{VLLM_HOST}:{VLLM_PORT}/health/"

    def is_vllm_server_alive():
        try:
            req = urllib.request.Request(HEALTH_URL, headers={"User-Agent": "CyberRottProbe"})
            with urllib.request.urlopen(req, timeout=2.0) as resp:
                return resp.status == 200
        except Exception:
            return False

    if is_vllm_server_alive():
        print(f"[✓] Servidor vLLM ativo detectado em {VLLM_HOST}:{VLLM_PORT}!")
    else:
        print(f"[*] Inicializando servidor vLLM em background na porta {VLLM_PORT} (PagedAttention + LoRA Sync)...")
        # Mata qualquer processo residual antigo na porta 8000 se houver
        !fuser -k 8000/tcp 2>/dev/null || true
        !nohup python -m trl.scripts.vllm_serve \
            --model microsoft/Phi-4-mini-instruct \
            --host 127.0.0.1 \
            --port 8000 \
            --gpu_memory_utilization 0.30 \
            --dtype bfloat16 > vllm_server.log 2>&1 &

        # Aguarda servidor inicializar completamente
        max_wait = 90
        poll_interval = 3
        start_wait = time.time()
        ready = False
        while time.time() - start_wait < max_wait:
            if is_vllm_server_alive():
                ready = True
                break
            elapsed = int(time.time() - start_wait)
            print(f"   Aguardando vLLM carregar os pesos na GPU... ({elapsed}s decorridos)")
            time.sleep(poll_interval)

        if ready:
            print(f"[✓] Servidor vLLM online e respondendo com sucesso em {VLLM_HOST}:{VLLM_PORT}!")
        else:
            print("[!] Aviso: Servidor vLLM demorou para responder. Verificando log recente:")
            !tail -n 30 vllm_server.log 2>/dev/null || true

    # 3. Execução do Treinamento Online GRPO v1.4 com vLLM Ativo e Auto-Resume
    print("\n" + "=" * 70)
    print("🚀 INICIANDO GRPO v1.4 COM ACELERAÇÃO vLLM (Tempo estimado: ~50-80 min na A100)")
    print("=" * 70 + "\n")

    !export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True && python scripts/train_grpo.py \
        --preset_v1_4 \
        --model_name_or_path microsoft/Phi-4-mini-instruct \
        --adapter_name_or_path ./CyberRott-v1.4-SFT-DoRA \
        --dataset_name data/cyberrott_final_train.jsonl \
        --anchor_file data/dataset_cyber_ready.jsonl \
        --output_dir ./CyberRott-v1.4-GRPO \
        --num_generations 8 \
        --max_prompt_length 768 \
        --max_completion_length 2048 \
        --learning_rate 5e-6 \
        --beta 0.02 \
        --per_device_train_batch_size 2 \
        --gradient_accumulation_steps 8 \
        --vllm_gpu_memory_utilization 0.30 \
        --vllm_server_host 127.0.0.1 \
        --vllm_server_port 8000 \
        --gradient_checkpointing \
        --save_steps 20 \
        --num_train_epochs 1 \
        --torch_dtype bfloat16 \
        --attn_implementation sdpa \
        --include_prm \
        --w_prm 0.25 \
        --use_vllm \
        --resume_from_checkpoint auto

    # 4. Encerramento limpo do servidor vLLM para liberar VRAM para os benchmarks subsequentes
    !pkill -f "trl.scripts.vllm_serve" 2>/dev/null || true

    # 5. Backup automático no Google Drive
    if os.path.exists("/content/drive/MyDrive/CyberRott_Checkpoints") and os.path.exists(LOCAL_GRPO_DIR):
        drive_dest = "/content/drive/MyDrive/CyberRott_Checkpoints/CyberRott-v1.4-GRPO"
        print(f"[*] Salvando backup do GRPO v1.4 no Google Drive: {drive_dest}...")
        shutil.copytree(LOCAL_GRPO_DIR, drive_dest, dirs_exist_ok=True)
        print("[✓] Backup do GRPO v1.4 concluído com sucesso!")


---
## 🎯 FASE 4: Inferência, Auditoria e Backup do Checkpoint Final GRPO v1.4

Nesta fase:
1. Carregamos o modelo otimizado pelo pipeline GRPO (`./CyberRott-v1.4-GRPO`) com fallback seguro para checkpoints anteriores.
2. Executamos um teste de auditoria técnica de cibersegurança em código C para verificar o raciocínio profundo de 7 estágios dentro de `<think>...</think>` e o formato conclusivo tripartite (`FATO:`, `DETECÇÃO:`, `SUGESTÃO DE CORREÇÃO:`).
3. Sincronizamos o checkpoint final com o Google Drive para persistência.


In [ ]:
import os
import shutil
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 1. Identificar o Checkpoint Final (Prioridade: GRPO v1.4 -> GRPO v1.3 -> SFT DoRA v1.4)
FINAL_CHECKPOINT = "./CyberRott-v1.4-GRPO"
if not (os.path.exists(FINAL_CHECKPOINT) and os.path.exists(os.path.join(FINAL_CHECKPOINT, "adapter_config.json"))):
    candidate_paths = [
        "/content/drive/MyDrive/CyberRott_Checkpoints/CyberRott-v1.4-GRPO",
        "/content/drive/MyDrive/CyberRott_Checkpoints/CyberRott_v1_4_GRPO",
        "./CyberRott-v1.4-SFT-DoRA",
        "/content/drive/MyDrive/CyberRott_Checkpoints/CyberRott-v1.4-SFT-DoRA",
        "/content/drive/MyDrive/CyberRott_Checkpoints/CyberRott-v1.3-GRPO",
        "./CyberRott-v1.3-GRPO",
    ]
    for candidate in candidate_paths:
        if os.path.exists(candidate) and os.path.exists(os.path.join(candidate, "adapter_config.json")):
            FINAL_CHECKPOINT = candidate
            break

print(f"[*] Checkpoint Selecionado: {FINAL_CHECKPOINT}")

BASE_MODEL = "microsoft/Phi-4-mini-instruct"
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float32

# Carrega Tokenizer e Modelo Base em bfloat16
tokenizer = AutoTokenizer.from_pretrained(
    FINAL_CHECKPOINT if os.path.exists(os.path.join(FINAL_CHECKPOINT, "tokenizer_config.json")) else BASE_MODEL,
    trust_remote_code=False
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else "cpu",
    trust_remote_code=False,
)

# Se houver checkpoint SFT intermediário, mescla o SFT antes do GRPO
sft_candidate = "./CyberRott-v1.4-SFT-DoRA"
if not os.path.exists(sft_candidate):
    sft_candidate = "/content/drive/MyDrive/CyberRott_Checkpoints/CyberRott-v1.4-SFT-DoRA"

if os.path.exists(sft_candidate) and os.path.exists(os.path.join(sft_candidate, "adapter_config.json")) and FINAL_CHECKPOINT != sft_candidate:
    print(f"[*] Mesclando adaptador SFT intermediário ({sft_candidate})...")
    base_model = PeftModel.from_pretrained(base_model, sft_candidate).merge_and_unload()

# Mescla pesos LoRA/DoRA finais com o modelo base para inferência direta
print(f"[*] Mesclando pesos adaptadores finais de {FINAL_CHECKPOINT}...")
model = PeftModel.from_pretrained(base_model, FINAL_CHECKPOINT).merge_and_unload()
model.eval()
print("[✓] Modelo CyberRott v1.4 GRPO pronto para auditoria técnica!")

# 2. Desafio de Auditoria em C (Buffer Overflow / Taint Analysis)
audit_prompt = (
    "Realize a auditoria de segurança rigorosa da seguinte função C em espaço de kernel/usuário:\n\n"
    "```c\n"
    "void parse_network_packet(const char *incoming_payload, int packet_len) {\n"
    "    char local_buffer[64];\n"
    "    if (packet_len > 0) {\n"
    "        strcpy(local_buffer, incoming_payload);\n"
    "    }\n"
    "    process_packet_header(local_buffer);\n"
    "}\n"
    "```\n\n"
    "Instruções:\n"
    "1. Desenvolva seu raciocínio analítico dentro da tag <think>...</think>.\n"
    "2. Declare a vulnerabilidade identificada, o CWE correspondente e o impacto.\n"
    "3. Forneça o patch corretivo defensivo com verificação rígida de limites."
)

messages = [
    {
        "role": "system",
        "content": "Você é o CyberRott, uma IA especialista em segurança ofensiva, defensiva e análise de vulnerabilidades com raciocínio analítico profundo."
    },
    {"role": "user", "content": audit_prompt}
]

formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

print("\n" + "=" * 70)
print("🛡️ RESPOSTA DO CYBERROTT (RACIOCÍNIO TRIPARTITE):")
print("=" * 70 + "\n")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        temperature=0.3,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

input_len = inputs["input_ids"].shape[1]
output_text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
print(output_text)

# 3. Backup para o Google Drive
DRIVE_BACKUP = "/content/drive/MyDrive/CyberRott_Checkpoints/CyberRott-v1.4-GRPO"
if os.path.exists("/content/drive/MyDrive") and os.path.exists(FINAL_CHECKPOINT):
    print(f"\n[*] Sincronizando checkpoint final para o Google Drive: {DRIVE_BACKUP}...")
    shutil.copytree(FINAL_CHECKPOINT, DRIVE_BACKUP, dirs_exist_ok=True)
    print("[✓] Backup no Google Drive concluído com sucesso!")


---
## 🏆 FASE 5: Avaliação Formal de Benchmark WMDP-Cyber com Test-Time Compute (TTC v1.4)

Nesta fase realizamos a avaliação oficial do CyberRott v1.4 no benchmark **WMDP-Cyber (1.987 questões)**:
1. Avalia a acurácia generativa com cadeia de pensamento (`<think>...</think>`) e extração de alternativa.
2. **Auto-Fusão BF16 Direta:** detecta os adaptadores LoRA/DoRA no Google Drive ou modelo standalone prévio, realizando a fusão pura em 16-bits (`bfloat16`) sem quantização.
3. **Test-Time Compute (TTC v1.4):** Utiliza Best-of-$N$ ($N=8$ rollouts paralelos) ou Majority Voting em 2048 tokens via motor **vLLM** em `bfloat16`.
4. Salva no Google Drive e faz download automático dos relatórios detalhados em `eval_results/` (`latest_summaryv1.4.json` e `latest_full_resultsv1.4.json`).


In [ ]:
# 0. Liberação de VRAM do kernel do Colab (evita segurar memória de treinos anteriores)
import gc
import torch
for _var in ["base_model", "model", "peft_model", "trainer"]:
    if _var in globals():
        del globals()[_var]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
print("[✓] VRAM do kernel do notebook limpa com sucesso!")

# 1. Montagem do Google Drive (para carregar checkpoints BF16 e salvar relatórios persistentes)
import os
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    print("[*] Montando Google Drive...")
    drive.mount("/content/drive")

# 2. Atualização dos scripts do repositório
!git pull origin main

# 3. Execução da avaliação com Auto-Fusão BF16 embutida e motor vLLM com Test-Time Compute (TTC)
!python scripts/eval_wmdp_vllm.py \
    --preset_v1_4 \
    --model_name ./CyberRott-v1.4-GRPO \
    --sft_adapter ./CyberRott-v1.4-SFT-DoRA \
    --num_samples 8 \
    --ttc_strategy best_of_n \
    --max_tokens 2048 \
    --output_dir eval_results

# 4. Download automático dos relatórios de avaliação mais recentes
from google.colab import files
for report_path in [
    "eval_results/latest_summaryv1.4.json",
    "eval_results/latest_full_resultsv1.4.json",
    "eval_results/latest_summary.json",
    "eval_results/latest_full_results.json"
]:
    if os.path.exists(report_path):
        print(f"[*] Baixando relatório: {report_path}...")
        files.download(report_path)


---
## 🌐 FASE 6: Publicação e Upload para o Hugging Face Hub (Model Card + Pesos Standalone)

Nesta etapa final, você publica o **CyberRott v1.4** completo no Hugging Face Hub (ou atualiza rapidamente apenas a documentação / Model Card):
1. **Autenticação Automática e Segura**:
   - Detecta o token do Hugging Face prioritariamente através dos **Secrets do Colab** (`HF_TOKEN`, ícone de chave 🔑 no menu lateral esquerdo).
   - Oferece fallback via prompt interativo seguro (`getpass`) caso o secret não esteja configurado.
2. **Sincronização de Documentação Oficial**:
   - Copia o `MODEL_CARD.md` técnico atualizado (com métricas completas do WMDP-Cyber, datasets e arquitetura v1.4) como `README.md` raiz do modelo.
   - Inclui o `Modelfile` para execução imediata no Ollama via `ollama run hf.co/<user>/CyberRott-v1.4-GRPO`.
3. **Upload Automatizado e Resiliente**:
   - O script `scripts/upload_to_hf.py` resolve dinamicamente os diretórios de pesos (`/content/CyberRott-v1.4-BF16-Standalone`, etc.) e gerencia o upload de safetensors com retries automáticos.
   - Suporta a flag `--readme-only` para atualizações de documentação em poucos segundos sem reenvio de pesos pesados.


In [ ]:
# ==============================================================================
# FASE 6: Upload Automático do CyberRott v1.4 para o Hugging Face Hub
# ==============================================================================

import os
import sys
import shutil

# 1. Obtenção Segura do Token Hugging Face (Secrets do Colab com fallback interativo)
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        print("[✓] Token Hugging Face recuperado dos Secrets do Google Colab (HF_TOKEN)!")
except Exception:
    pass

if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    from getpass import getpass
    print("[!] DICA: Para não digitar seu token a cada execução, adicione o secret 'HF_TOKEN' no menu lateral esquerdo (ícone 🔑).")
    hf_token = getpass("Insira seu Token Hugging Face (Write): ").strip()

os.environ["HF_TOKEN"] = hf_token

# 2. Atualizar repositório local e sincronizar o MODEL_CARD.md mais recente
!git pull origin main

# 3. Verificação de pesos Standalone (Fusão automática de segurança se a Fase 5 foi pulada)
standalone_dir = "/content/CyberRott-v1.4-BF16-Standalone" if os.path.exists("/content") else "./CyberRott-v1.4-BF16-Standalone"
if not (os.path.exists(standalone_dir) and any(f.endswith('.safetensors') for f in os.listdir(standalone_dir))):
    print("[*] Standalone BF16 não encontrado. Executando fusão com scripts/merge_lora.py...")
    !python scripts/merge_lora.py \
        --base_model microsoft/Phi-4-mini-instruct \
        --sft_adapter ./CyberRott-v1.4-SFT-DoRA \
        --adapter ./CyberRott-v1.4-GRPO \
        --output {standalone_dir} \
        --torch_dtype bfloat16

# 4. Escolha do Modo de Upload:
# Defina README_ONLY = True se quiser atualizar APENAS o Model Card / documentação em segundos.
# Defina README_ONLY = False para enviar a pasta completa de pesos Standalone (Safetensors + Configs).
README_ONLY = False
REPO_NAME = "CyberRott-v1.4-GRPO"

# 5. Execução do Script de Upload
cmd = f"python scripts/upload_to_hf.py --repo_name {REPO_NAME}"
if README_ONLY:
    cmd += " --readme-only"

print(f"[*] Executando: {cmd}")
!{cmd}
